# 12 — Risk aggregation + Kepler package + viewer

Aggregate per-chip risk to plant alerts and export assets for the rednet-risk-viewer / Kepler.


In [6]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first)."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check, cwd=REPO_ROOT)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = REPO_ROOT / c
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

OUT_ROOT = REPO_ROOT / "deployment/outputs/by_plant"
print("OUT_ROOT exists:", OUT_ROOT.exists())

if OUT_ROOT.exists():
    plants = [
        p for p in OUT_ROOT.iterdir()
        if p.is_dir() and "(test)" not in p.name.lower()
    ]
    print("plants:", len(plants))
    print("plant dirs:")
    for p in plants:
        print(" -", p.name)



REPO_ROOT: /Users/ameerfiras/REDNET-ML
OUT_ROOT exists: True
plants: 4
plant dirs:
 - osm_way_386838289
 - osm_way_1236881046
 - osm_way_1079022886
 - osm_way_449632054


## 12.1 Risk-field scripts


In [9]:

for s in [
    "hab_aggregate.py",
    "build_plant_risk_json.py",
    "hab_transport_trips.py",
]:
    p = REPO_ROOT / "deployment/risk_field" / s
    print(s, "->", "OK" if p.exists() else "MISSING")
    if p.exists():
        sh(f'python "{p}" --help', check=False)


hab_aggregate.py -> OK

▶ python "/Users/ameerfiras/REDNET-ML/deployment/risk_field/hab_aggregate.py" --help
usage: Aggregate HAB particle points into density + envelope
       [-h] --in_points_geojson IN_POINTS_GEOJSON --out_density_geojson
       OUT_DENSITY_GEOJSON --out_envelope_geojson OUT_ENVELOPE_GEOJSON
       [--grid_km GRID_KM] [--cap_points CAP_POINTS] [--buffer_m BUFFER_M]
       [--simplify_m SIMPLIFY_M]

options:
  -h, --help            show this help message and exit
  --in_points_geojson IN_POINTS_GEOJSON
  --out_density_geojson OUT_DENSITY_GEOJSON
  --out_envelope_geojson OUT_ENVELOPE_GEOJSON
  --grid_km GRID_KM     Density grid cell size in km
  --cap_points CAP_POINTS
                        Max points used for density (subsample if bigger).
                        0=disable
  --buffer_m BUFFER_M   Envelope buffer radius around points (meters)
  --simplify_m SIMPLIFY_M
                        Envelope simplification tolerance (meters)
build_plant_risk_json.py -> OK



## 12.2 Kepler package


In [13]:

p = REPO_ROOT / "deployment/risk_field/make_kepler_package.py"
print("Exists:", p.exists())

if p.exists():
    sh(f'python "{p}" --help', check=False)

# Example:
# sh(
#   f'python "{p}" '
#   f'--root_dir "{REPO_ROOT / "deployment/outputs/by_plant"}" '
#   f'--out "{REPO_ROOT / "deployment/outputs/kepler_package.zip"}"'
# )


Exists: True

▶ python "/Users/ameerfiras/REDNET-ML/deployment/risk_field/make_kepler_package.py" --help
usage: Make a single Kepler-ready GeoJSON: trips + points + density + envelope
       [-h] --in_csv IN_CSV --plant_lat PLANT_LAT --plant_lon PLANT_LON
       --out_geojson OUT_GEOJSON [--window_km WINDOW_KM]
       [--step_hours STEP_HOURS] [--hycom_stride HYCOM_STRIDE]
       [--max_particles MAX_PARTICLES] [--seed_per_obs SEED_PER_OBS]
       [--seed_floor SEED_FLOOR]
       [--min_seeds_if_positive MIN_SEEDS_IF_POSITIVE]
       [--half_life_days HALF_LIFE_DAYS] [--min_survival MIN_SURVIVAL]
       [--base_diff_m BASE_DIFF_M] [--risk_diff_m RISK_DIFF_M]
       [--max_drift_m MAX_DRIFT_M] [--grid_km GRID_KM]
       [--cap_points CAP_POINTS] [--buffer_m BUFFER_M]
       [--simplify_m SIMPLIFY_M] [--hycom_cache_npz HYCOM_CACHE_NPZ]

options:
  -h, --help            show this help message and exit
  --in_csv IN_CSV       inference_all_months.csv
  --plant_lat PLANT_LAT
  --plant_lon P

## 12.3 Viewer commands


In [ ]:

# cd rednet-risk-viewer
# npm install
# npm run dev
